# Plotly Dash dashboard

Rodrigo Ariel Quilodran. IBM historical course dataset. AI-assisted code, executed and verified.

In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA, OUT, FIG = ROOT/'data', ROOT/'results', ROOT/'figures'
plt.rcParams.update({'figure.figsize': (10, 5), 'font.size': 12, 'axes.spines.top': False, 'axes.spines.right': False})


## App and callbacks
The dropdown controls the site. The RangeSlider controls the payload interval. Both inputs update both charts through component IDs.

In [2]:
import sys,inspect
sys.path.insert(0,str(ROOT))
import dashboard
print(inspect.getsource(dashboard.update))

@app.callback(Output('success-pie-chart','figure'),Output('success-payload-scatter-chart','figure'),Output('filter-summary','children'),Input('site-dropdown','value'),Input('payload-slider','value'))
def update(site,payload):
 filtered=df[df['Payload Mass (kg)'].between(*payload)]
 if site!='ALL':filtered=filtered[filtered['Launch Site'].eq(site)]
 if site=='ALL':
  counts=filtered.groupby('Launch Site',as_index=False)['class'].sum();pie=px.pie(counts,values='class',names='Launch Site',title='Share of successful landings by site')
 else:
  counts=filtered['class'].value_counts().reindex([0,1],fill_value=0).rename_axis('Class').reset_index(name='Count');counts['Outcome']=counts.Class.map({0:'Class 0',1:'Class 1'})
  pie=px.pie(counts,values='Count',names='Outcome',title=f'{site}: landing success and failure')
 scatter=px.scatter(filtered,x='Payload Mass (kg)',y='class',color='Booster Version Category',hover_data=['Launch Site','Flight Number'],title='Payload and landing outcome by boost

## Meaningful filter checks
Compare the unfiltered snapshot, KSC and a narrower payload range. These same states were inspected in Chrome.

In [3]:
for site,limits,expected in [('ALL',[0,10000],56),('KSC LC-39A',[0,10000],13),('KSC LC-39A',[0,6000],11)]:
    pie,scatter,summary=dashboard.update(site,limits)
    print(summary)
    assert summary.startswith(str(expected)+' launches')
    assert sum(len(t.x) for t in scatter.data)==expected
print('All three callback checks passed')
print('Run: python dashboard.py; browse http://127.0.0.1:8050')

56 launches shown; 24 successful. Payload 0 to 10,000 kg.
13 launches shown; 10 successful. Payload 0 to 10,000 kg.


11 launches shown; 10 successful. Payload 0 to 6,000 kg.
All three callback checks passed
Run: python dashboard.py; browse http://127.0.0.1:8050
